# Ecosistema de procesamiento paralelo para datos masivos

## Módulo 3. Lección 2: Programación MapReduce con lenguajes de alto nivel: Hive.

### Ejercicios resueltos 1

**IMPORTANTE**

Ejecuta las celdas en orden para evitar errores de ejecución. Esto garantiza que todos los archivos necesarios se creen y se ubiquen correctamente a lo largo del proceso.

#### Obtención de los datos

En caso de no tener el fichero de la lección anterior, lo debemos descargar y subir a HDFS.

In [1]:
import os

# creamos la carpeta para guardar el fichero
! mkdir -p /media/notebooks/argentina
os.chdir("/media/notebooks/argentina")
! pwd

/media/notebooks/argentina


In [2]:
# Descargamos el fichero 
! wget https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv \
-O /media/notebooks/argentina/convocatorias-2020.csv 

--2024-08-26 11:52:41--  https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv [following]
--2024-08-26 11:52:42--  https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5675992 (5.4M) [text/plain]
Saving to: ‘/media/notebooks/argentina/convocatorias-2020.csv’

/media/notebooks/ar 100%[===================>]   5.41M  7.76MB/s    in 0.7s    

2024-08-26 11:52:43 (7.76 MB/s) - ‘/media/notebooks/argentina/convocatorias-2020.c

In [11]:
# subimos el fichero a HDFS 
! hdfs dfs -mkdir /argentina
! hdfs dfs -mkdir /argentina/convocatorias
! hdfs dfs -put /media/notebooks/argentina/convocatorias-2020.csv /argentina/convocatorias
! hdfs dfs -ls /argentina/convocatorias

mkdir: `/argentina': File exists
mkdir: `/argentina/convocatorias': File exists
Found 1 items
-rw-r--r--   3 root supergroup    5675992 2024-08-28 08:56 /argentina/convocatorias/convocatorias-2020.csv


#### Creación carpeta para los ejercicios

In [2]:
# creamos la carpeta para los ejercicios en local
! mkdir -p /media/notebooks/argentina/hive

# nos vamos a esa carpeta y comprobamos donde estamos
os.chdir("/media/notebooks/argentina/hive")
! pwd

/media/notebooks/argentina/hive


#### Dataframe de pandas para comprender los datos y comparar la salida

Utilizaremos un dataframe de pandas para ayudarnos a entender los datos y también podemos comprobar los ejercicios comparando la salida dada y la salida esperada.

In [36]:
import pandas as pd

# evitamos la notación científica en todos los ejercicios
pd.set_option('display.float_format', '{:.2f}'.format)

# dataframe convocatorias
columnas = [
             'numero_procedimiento', 'nro_saf', 'descripcion_saf', 'nro_uoc', 
             'descripcion_uoc','tipo_de_procedimiento', 'modalidad', 'apartado_directa', 
             'ejercicio', 'fecha_de_publicacion', 'fecha_de_apertura', 'etapa', 'alcance', 
             'nombre_del_procedimiento', 'objeto_del_procedimiento', 'monto_estimado', 'tipo_de_operacion'
           ]

df_convocatorias = pd.read_csv('/media/notebooks/argentina/convocatorias-2020.csv', names=columnas) 
df_convocatorias.head(3)

,numero_procedimiento,nro_saf,descripcion_saf,nro_uoc,descripcion_uoc,tipo_de_procedimiento,modalidad,apartado_directa,ejercicio,fecha_de_publicacion,fecha_de_apertura,etapa,alcance,nombre_del_procedimiento,objeto_del_procedimiento,monto_estimado,tipo_de_operacion
0,40/51-0159-LPR19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Licitacion Privada,Sin Modalidad,NaN,2019,02/01/2020 08:00:00 a.m.,10/01/2020 10:00:00 a.m.,Única,Nacional,“Adquisición de Elementos de Librería para el ...,“Adquisición de Elementos de Librería para el ...,1957142.84,Proceso de Compra
1,22-0041-CDI19,250,250 - Caja de Retiros Jubilaciones y Pensiones...,22,22 - Dpto de Compras y Suministros - Caja de R...,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 09:00:00 a.m.,Única,Nacional,Servicio de seguridad de información sensible ...,Servicio de seguridad de información sensible ...,119915.25,Proceso de Compra
2,40/51-0918-CDI19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 10:00:00 a.m.,Única,Nacional,“Servicio Médico Asistencial para Organismos V...,“Servicio Médico Asistencial para Organismos V...,99000.00,Proceso de Compra


#### **Ejercicio 01.** Creación de una tabla externa. Convocatorias 2020.

Lo primero que debemos hacer es crear la tabla con la que realizaremos los ejercicios de esta sección. Para ello es necesario conocer los tipos que tenemos de datos y crear el fichero de consulta adecuado.

In [4]:
%%writefile tabla-convocatorias.hql
CREATE EXTERNAL TABLE convocatorias (
     numero_procedimiento STRING,
     nro_saf INT,
     descripcion_saf STRING,
     nro_uoc INT,
     descripcion_uoc STRING,
     tipo_de_procedimiento STRING,
     modalidad STRING,
     apartado_directa STRING,
     ejercicio INT,
     fecha_de_publicacion STRING,
     fecha_de_apertura STRING,
     etapa STRING,
     alcance STRING,
     nombre_del_procedimiento STRING,
     objeto_del_procedimiento STRING,
     monto_estimado FLOAT,
     tipo_de_operacion STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/argentina/convocatorias/'
TBLPROPERTIES ("skip.header.line.count"="0");

Writing tabla-convocatorias.hql


In [5]:
# ejecucion del fichero hql mediante beeline
!beeline -u "jdbc:hive2://" -f tabla-convocatorias.hql

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [8]:
#mostramos los campos
!beeline -u "jdbc:hive2://" -e "describe convocatorias;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [12]:
#mostramos las tres primeas filas
!beeline -u "jdbc:hive2://" -e \
"SELECT numero_procedimiento, nro_saf, descripcion_saf \
FROM convocatorias \
LIMIT 3;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

#### **Ejercicio 02.** Extracción de valores únicos. Descripcion SAF.

El primer ejercicio consiste en contar el número de valores únicos de la columna que contiene las descripciones del Servicio Administrativo Financiero (Descripcion SAF). De esa manera, sabremos exactamente de cuántos tipos de SAF tenemos para los procedimientos de las convocatorias.

Necesitaremos el campo **descripcion_saf** de la tabla convocatorias y la función COUNT(*), en este caso, contaremos el número de descripciones diferentes, por lo que utilizaremos **COUNT(DISTINCT campo)**, para obtener el número de valores únicos para el campo especificado.

In [10]:
# campos únicos
!beeline -u "jdbc:hive2://" -e \
"SELECT COUNT(DISTINCT descripcion_saf) as num_descripciones \
FROM convocatorias;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [11]:
# comprobamos en el dataframe el número de valores únicos para esa columna
num = df_convocatorias['descripcion_saf'].nunique()
print(f"\nValores en el dataframe:\t{num}")


Valores en el dataframe:	125


#### **Ejercicio 03.** Búsqueda del valor máximo. Mayor presupuesto estimado.

Este tercer ejercicio consiste en extraer el máximo presupuesto estimado. Utilizaremos el campo monto_estimado de la tabla convocatorias y la función MAX(campo), donde campo es el campo especificado.

In [13]:
# máximo monto estimado
!beeline -u "jdbc:hive2://" -e \
"SELECT CAST((MAX(monto_estimado)) AS DECIMAL(20, 2)) as monto_max \
FROM convocatorias;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [15]:
# comprobamos en el dataframe
maximo = df_convocatorias['monto_estimado'].max()
print(f"\nMáximo valor para el monto estimado:\t{maximo}")


Máximo valor para el monto estimado:	8288160000.0


#### **Ejercicio 04.** Búsqueda del valor medio. Presupuesto oficial estimado para obras nacionales e internacionales.

En el ejercicio número 4 vamos a calcular cuál es el presupuesto medio que se emplea en las obras públicas para aquellas que son nacionales y las que son internacionales. Este mismo ejercicio se realizó en la lección anterior, lo vamos a repetir con el fin de realizar una comparación exacta entre ambos procedimientos.

Los campos ahora que intervienen en la consulta son monto_estimado y alcance. Debemos utilizar la cláusula 'GROUP BY' para agrupar por alcance nacional e internacional, y así realizar la media para ambos valores.

Debemos recordar que queríamos ignorar los valores 'erróneos' de monto estimado, es decir, aquellos que son 0. Lo haremos mediante la cláusula WHERE donde filtraremos los valores de monto estimado.

In [18]:
# monto medio para alcance nacional e internacional
!beeline -u "jdbc:hive2://" -e \
"SELECT alcance, CAST((AVG(monto_estimado)) AS DECIMAL(20, 2)) as media_presupuesto \
FROM convocatorias \
WHERE monto_estimado != 0 \
GROUP BY alcance;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [24]:
# comprobamos en el dataframe
# eliminamos los que tienen valor 0
df_filtrado = df_convocatorias[df_convocatorias['monto_estimado'] != 0]
# agrupamos y calculamos la media
promedio_monto_por_alcance = df_filtrado.groupby('alcance')['monto_estimado'].mean()
print(f"\nValores en el dataframe:\n\n{promedio_monto_por_alcance}")


Valores en el dataframe:

alcance
Internacional   12112060.49
Nacional         8868861.54
Name: monto_estimado, dtype: float64


#### **Ejercicio 05.** Conteo por rangos. Número de procedimientos por año.
    
En este ejercicio realizaremos un conteo del número de procedimientos que ha habido en cada ejercicio, es decir, en cada año.

El campo que necesitamos es ejercicio. De nuevo, utilizaremos la cláusula 'GROUP BY' para agrupar por el año, y así realizar el conteo

In [16]:
# procedimientos por ejercicios
!beeline -u "jdbc:hive2://" -e \
"SELECT ejercicio, COUNT(*) as num_procedimientos \
FROM convocatorias \
GROUP BY ejercicio;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [10]:
# comprobamos en el dataframe
df_ejercicios = df_convocatorias.groupby('ejercicio')['numero_procedimiento'].count()
print(f"Número de procedimientos por año:")
df_ejercicios

Número de procedimientos por año:


ejercicio
2017        2
2019      587
2020    12960
Name: numero_procedimiento, dtype: int64

#### **Ejercicio 06.** Tipos de procedimientos. Transparencia de compras.

Este ejercicio va a evaluar la transparencia en el sistema de compras del Estado procesados en el sistema COMPRAR. Para ello, debemos tener claro el criterio que seguiremos: El sistema es más transparente si se utiliza el procedimiento de licitación pública. Para contestar a ésta pregunta calcularemos el porcentaje de cada tipo de procedimiento.

El campo que necesitamos es tipo_de_procedimiento. Para calcular los porcentajes necesitamos calcular primero el número total de procedimientos, lo hacemos mediante una subconsulta para ese campo utilizando COUNT(*). Para aumentar la eficiencia y que solo calcule el total de los campos una vez, utilizaremos WITH, creando así una tabla temporal con el resultado de contar los registros.

In [28]:
# porcentajes de tipos de procedimientos
!beeline -u "jdbc:hive2://" -e \
"WITH total_registros AS ( \
     SELECT COUNT(*) AS total FROM convocatorias \
) \
SELECT tipo_de_procedimiento, \
     CAST((COUNT(*) * 100.0 / total_registros.total) AS DECIMAL(20, 2)) AS porcentaje \
FROM convocatorias, total_registros \
GROUP BY tipo_de_procedimiento, total_registros.total \
ORDER BY porcentaje DESC;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://
Hive Session

In [39]:
# comprobamos en el dataframe
totales = df_convocatorias['tipo_de_procedimiento'].value_counts()

# reemplazamos por no diponible los nulos y los vacíos
df_convocatorias['tipo_de_procedimiento'].replace('', 'no disponible', inplace=True)
df_convocatorias['tipo_de_procedimiento'].fillna('no disponible', inplace=True)

# calculamos los porcentajes
porcentaje_tipos = (totales / len(df_convocatorias)) * 100
print(f"\nValores en el dataframe:\n\n{porcentaje_tipos}")


Valores en el dataframe:

tipo_de_procedimiento
Contratación Directa   49.89
Licitacion Privada     38.17
Licitacion Pública     10.93
Subasta                 0.47
Concurso Privado        0.27
Concurso Público        0.17
no disponible           0.10
Name: count, dtype: float64
